# n-gram

An n-gram is a sequence of n tokens (typically words) for some integer n. The NGram class can be used to transform input features into n-grams.

NGram takes as input a sequence of strings (e.g. the output of a Tokenizer). The parameter n is used to determine the number of terms in each n-gram. The output will consist of a sequence of n-grams where each n-gram is represented by a space-delimited string of n consecutive words. If the input sequence contains fewer than n strings, no output is produced.

## Create PySpark environment and import library

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local').appName('MyNgram').getOrCreate()

from pyspark.ml.feature import Tokenizer, RegexTokenizer, NGram
from pyspark.sql.functions import col, udf
from pyspark.sql.types import IntegerType

## 3-gram from words df

In [2]:
wordDataFrame = spark.createDataFrame([
    (0, ['Hi', 'I', 'heard', 'about', 'Spark']), 
    (1, ['I', 'wish', 'python', 'could', 'use', 'case', 'classes']),
    (2, ['Logistic', 'regression', 'models', 'are', 'neat'])
], ['id', 'words'])

wordDataFrame.show()

ngram = NGram(inputCol = 'words', outputCol = 'grams', n = 3)
ngram.transform(wordDataFrame).show()

# select grams only
ngram.transform(wordDataFrame).select('grams').show(truncate=False)

+---+--------------------+
| id|               words|
+---+--------------------+
|  0|[Hi, I, heard, ab...|
|  1|[I, wish, python,...|
|  2|[Logistic, regres...|
+---+--------------------+

+---+--------------------+--------------------+
| id|               words|               grams|
+---+--------------------+--------------------+
|  0|[Hi, I, heard, ab...|[Hi I heard, I he...|
|  1|[I, wish, python,...|[I wish python, w...|
|  2|[Logistic, regres...|[Logistic regress...|
+---+--------------------+--------------------+

+--------------------------------------------------------------------------------------+
|grams                                                                                 |
+--------------------------------------------------------------------------------------+
|[Hi I heard, I heard about, heard about Spark]                                        |
|[I wish python, wish python could, python could use, could use case, use case classes]|
|[Logistic regression models

## Process text file input

First tokenize, then 3-gram

In [3]:
df_text = spark.read.text("data/1.txt")

df_text.show()

tokenizer = Tokenizer(inputCol = 'value', outputCol = 'words')
tokenized = tokenizer.transform(df_text)

ngram = NGram(inputCol = 'words', outputCol = 'grams', n = 3)
ngram.transform(tokenized).show(truncate=True)

#ngram.transform(tokenized).select('grams').show(truncate=False)

+--------------------+
|               value|
+--------------------+
|0001 Lorem Ipsum ...|
|0002 Contrary to ...|
|0003 The standard...|
|0004 It is a long...|
|0005 There are ma...|
|0006 Lorem ipsum ...|
|0007 Mauris ut da...|
|0008 Lorem Ipsum ...|
|0009 Contrary to ...|
|0010 The standard...|
|0011 It is a long...|
|0012 There are ma...|
+--------------------+

+--------------------+--------------------+--------------------+
|               value|               words|               grams|
+--------------------+--------------------+--------------------+
|0001 Lorem Ipsum ...|[0001, lorem, ips...|[0001 lorem ipsum...|
|0002 Contrary to ...|[0002, contrary, ...|[0002 contrary to...|
|0003 The standard...|[0003, the, stand...|[0003 the standar...|
|0004 It is a long...|[0004, it, is, a,...|[0004 it is, it i...|
|0005 There are ma...|[0005, there, are...|[0005 there are, ...|
|0006 Lorem ipsum ...|[0006, lorem, ips...|[0006 lorem ipsum...|
|0007 Mauris ut da...|[0007, mauris, ut...|[00

In [7]:
spark.stop()